# CatBoost 実践

カテゴリ変数を自動処理し、ordered boosting でリークを抑える CatBoost の使い方を学びます。

> **Kaggle Tutorial** 対応ノートブック（`/learn/gradient-boosting/06-catboost`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# [Colab / ローカル環境用セットアップ] catboost が未インストールの場合は自動インストールします
import subprocess
import sys

try:
    import catboost  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost"], check=True)

In [ ]:
# [Colab / ローカル環境用セットアップ]
# Kaggle Notebook 上では既存の /kaggle/input/titanic をそのまま使用し、
# Google Colab やローカル環境では互換スキーマの Titanic サンプルデータ（891行 / 418行）を自動配置します。
# ※ Kaggle 本番データで提出する場合は kaggle CLI (!kaggle competitions download -c titanic) をご利用ください。
import os
import numpy as np
import pandas as pd

def _ensure_titanic_data():
    target_dir = "/kaggle/input/titanic"
    if os.path.exists(os.path.join(target_dir, "train.csv")):
        return

    try:
        os.makedirs(target_dir, exist_ok=True)
        data_dir = target_dir
    except OSError:
        data_dir = os.path.abspath("./.kaggle_input/titanic")
        os.makedirs(data_dir, exist_ok=True)
        _orig_read_csv = pd.read_csv

        def _compat_read_csv(filepath_or_buffer, *args, **kwargs):
            if isinstance(filepath_or_buffer, str) and filepath_or_buffer.startswith("/kaggle/input/"):
                rel = filepath_or_buffer[len("/kaggle/input/"):]
                local_path = os.path.abspath(os.path.join("./.kaggle_input", rel))
                if os.path.exists(local_path):
                    filepath_or_buffer = local_path
            return _orig_read_csv(filepath_or_buffer, *args, **kwargs)

        pd.read_csv = _compat_read_csv

    rng = np.random.default_rng(42)
    surnames = ["Braund", "Cumings", "Heikkinen", "Futrelle", "Allen", "Moran", "McCarthy", "Palsson", "Johnson", "Nasser"]
    given_m = ["Owen", "William", "James", "Henry", "Leonard", "Timothy", "charles", "George"]
    given_f = ["Florence", "Laina", "Lily", "elisabeth", "Adele", " Marguerite", "Anna", "Mary"]

    def _make_split(n, start_id, with_target):
        pclass = rng.choice([1, 2, 3], size=n, p=[0.24, 0.21, 0.55])
        sex = rng.choice(["male", "female"], size=n, p=[0.65, 0.35])
        age = np.clip(rng.normal(29.7, 14.5, size=n), 0.42, 80.0).round(1)
        sibsp = rng.choice([0, 1, 2, 3, 4], size=n, p=[0.68, 0.23, 0.05, 0.02, 0.02])
        parch = rng.choice([0, 1, 2, 3], size=n, p=[0.76, 0.13, 0.09, 0.02])
        base_fare = np.where(pclass == 1, 60.0, np.where(pclass == 2, 20.0, 11.0))
        fare = (base_fare * rng.lognormal(0.0, 0.55, size=n)).round(2)
        embarked = rng.choice(["S", "C", "Q"], size=n, p=[0.72, 0.19, 0.09]).astype(object)

        names = []
        cabins = []
        tickets = []
        for i in range(n):
            sn = surnames[i % len(surnames)]
            if sex[i] == "male":
                title = "Master" if age[i] < 13 else "Mr"
                gn = given_m[i % len(given_m)].strip().capitalize()
            else:
                title = "Miss" if age[i] < 24 else "Mrs"
                gn = given_f[i % len(given_f)].strip().capitalize()
            names.append(f"{sn}, {title}. {gn}")
            cabins.append(f"{'ABC'[pclass[i] - 1]}{10 + (i % 90)}" if rng.random() > 0.77 else np.nan)
            tickets.append(f"PC {17500 + (i % 350)}" if pclass[i] == 1 else str(230000 + i))

        age_obj = age.astype(float)
        age_obj[rng.random(n) < 0.20] = np.nan
        embarked[rng.random(n) < 0.005] = np.nan
        if not with_target:
            fare[0] = np.nan

        data = {
            "PassengerId": np.arange(start_id, start_id + n),
            "Pclass": pclass,
            "Name": names,
            "Sex": sex,
            "Age": age_obj,
            "SibSp": sibsp,
            "Parch": parch,
            "Ticket": tickets,
            "Fare": fare,
            "Cabin": pd.Series(cabins, dtype=object),
            "Embarked": pd.Series(embarked, dtype=object),
        }
        if with_target:
            family_size = sibsp + parch + 1
            logit = (
                -1.65
                + np.where(sex == "female", 2.45, 0.0)
                + np.where(pclass == 1, 1.35, np.where(pclass == 2, 0.35, -0.55))
                + np.where(np.nan_to_num(age_obj, nan=29.0) < 12, 1.2, 0.0)
                + np.where((family_size >= 2) & (family_size <= 4), 0.75, -0.35)
            )
            prob = 1.0 / (1.0 + np.exp(-logit))
            survived = (rng.random(n) < prob).astype(int)
            ordered = {"PassengerId": data["PassengerId"], "Survived": survived}
            for k, v in data.items():
                if k != "PassengerId":
                    ordered[k] = v
            return pd.DataFrame(ordered)
        return pd.DataFrame(data)

    _make_split(891, 1, True).to_csv(os.path.join(data_dir, "train.csv"), index=False)
    _make_split(418, 892, False).to_csv(os.path.join(data_dir, "test.csv"), index=False)
    print(f"Titanic dataset ready at: {data_dir}")

_ensure_titanic_data()

In [ ]:
# カテゴリ列の欠損を定数文字列 "Missing" に統一した上で訓練・検証へ層化分割します
from sklearn.model_selection import train_test_split

df = pd.read_csv("/kaggle/input/titanic/train.csv")
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = df[features].copy()
X["Sex"] = X["Sex"].astype(str)
X["Embarked"] = X["Embarked"].fillna("Missing").astype(str)
y = df["Survived"]
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## CatBoost とは

CatBoost（Yandex 製）は、**カテゴリ変数の扱いに優れた** 勾配ブースティング（GBDT） 実装です。
前処理が少なく、デフォルト設定でも強いのが特徴で、カテゴリの多いデータで威力を発揮します。

## カテゴリ変数の自動処理

ラベルエンコーディングやターゲットエンコーディングを手で書かなくても、
列を指定するだけで CatBoost が内部で適切に数値化します。

In [ ]:
from catboost import CatBoostClassifier

cat_features = ["Sex", "Embarked", "Pclass"]   # カテゴリ列を指定するだけ

model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    eval_metric="AUC",
    random_seed=42,
    verbose=0,
)
model.fit(
    X_tr, y_tr,
    cat_features=cat_features,
    eval_set=(X_va, y_va),
    early_stopping_rounds=100,
)

In [ ]:
from sklearn.metrics import roc_auc_score
val_auc = roc_auc_score(y_va, model.predict_proba(X_va)[:, 1])
print(f"CatBoost Validation AUC: {val_auc:.4f} (best_iteration={model.get_best_iteration()})")

## ordered boosting でリークを抑える

CatBoost のターゲットエンコーディングは、各サンプルの統計量を
**そのサンプルより前の行だけ**から計算する「ordered」方式を使い、
通常のターゲットエンコードで起きがちなデータリークを構造的に抑えます。

> **Tip:** CatBoost は欠損やカテゴリをそのまま渡せるため、**前処理を最小化して素早くベースライン**を 作れます。カテゴリが多いコンペではまず CatBoost を試す価値があります。

> **Note:** 3 つの GBDT はパラメータ名が少しずつ違います（木の深さ＝XGBoost/CatBoost は depth、 LightGBM は num_leaves など）。次のレッスンで対応関係を整理します。

## 次のステップ

次は、3 大 GBDT の比較・使い分けと、共通する重要パラメータを整理します。